In [1]:
# necessary modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from highlight_text import ax_text, fig_text
from drawarrow import ax_arrow, fig_arrow
from pyfonts import load_google_font
import matplotlib.patches as patches
from pywaffle import Waffle
import math

import warnings
warnings.filterwarnings("ignore")

import os

In [3]:
# Nominations data
data_nominee = pd.read_csv(
    os.path.join(
        "..", "data", 
        "Nobel Peace Nominees (Decade).csv"
    )
)

df_nominee = data_nominee.copy()

# ========================================= #

# Winners data
data_winners = pd.read_csv(
    os.path.join(
        "..", "data", 
        "Nobel Peace Winners (Decade).csv"
    )
)

df_winners = data_winners.copy()

In [4]:
# transform data for easier plotting
df_nominee_melt = df_nominee.melt(
    id_vars=["Decade"], 
    value_vars=["Africa", "America", "Asia", "Europe", "Oceania"],
    var_name="region", 
    value_name="count"
)

# maximim area under cultivation & the decade
max_decade = (df_nominee[df_nominee["Total"] == df_nominee["Total"].max()]["Decade"]).values[0]
max_decade_count = (df_nominee[df_nominee["Total"] == df_nominee["Total"].max()]["Total"]).values[0]

# labels for each foodgrain
region_labels = df_nominee_melt["region"].unique().tolist()

In [8]:
# CUSTOMIZATIONS

# fonts
mw_reg = load_google_font("Merriweather Sans")
mw_i = load_google_font("Merriweather Sans", italic=True)
mw_b = load_google_font("Merriweather Sans", weight=500)
mw_bi = load_google_font("Merriweather Sans", weight=500, italic=True)

# colors
colors = [
    "#212529",  # Africa
    "#e63946",  # America
    "#ffd60a",  # Asia
    "#023e8a",  # Europe
    "#6a994e",  # Oceania
]
bg = "#fbfffe"
annot_color = "#092328"

colors.append(bg)

# ANNOTATIONS

title_main = "Nobel Peace Nominations - From Europe to the World"
title_sub = "Nobel Peace Prize nominations shifted from overwhelming European dominance\nto a far more global—and increasingly Asian—landscape"
data_source = "Data: Wikipedia; nobelprize.org"

In [6]:
# PLOT SETTINGS

ncols = df_nominee_melt["Decade"].nunique()
dpi = 300
save_name = "nobel_peace.jpeg"

In [9]:
# plotting

fig, axs = plt.subplots(ncols=ncols, figsize=(30, 24), dpi=dpi)

fig.subplots_adjust(left=0.1, right=0.9, top=0.75, bottom=0.1)
fig.set_facecolor(bg)


for year, ax in zip(df_nominee_melt["Decade"].unique(), axs):
    
    values = list(df_nominee_melt[df_nominee_melt["Decade"] == year]["count"].values)
    values.append(max_decade_count - sum(values))
    
    Waffle.make_waffle(
    ax=ax,
    rows=80,
    columns=20,
    values=values,
    vertical=True,
    colors=colors,
    block_edge_width=0,
    interval_ratio_x=0.1,
    interval_ratio_y=0.1
  )
  
    ax.text(x=0.1, y=-0.03, s=str(year)+"s", fontsize=23, ha="center", fontweight="bold", font=mw_b, color=annot_color)


# legend
legend_xy = [(0.28, 0.02), (0.38, 0.02), (0.48, 0.02), (0.58, 0.02), (0.68, 0.02)]
legend_colors = colors[:-1]
legend_text = ["Africa", "America", "Asia", "Europe", "Oceania"]
box_w, box_h = 0.03, 0.02
for xy, c, text in zip(legend_xy, legend_colors, legend_text):
    legend = patches.Rectangle(
        (xy[0], xy[1]- box_h/2), width=box_w, height=box_h,
        transform=fig.transFigure,
        edgecolor=annot_color, facecolor=c,
        linewidth=0.8, alpha=1,
        clip_on=False, zorder=10
    )
    ax.add_patch(legend)
    fig_text(
        x=xy[0]+box_w/2, y=(xy[1])-0.018, s=text,
        size=25, color=annot_color, font=mw_b,
        ha="center", alpha=0.9
    )



winner_region_cols = ["Africa", "America", "Asia", "Europe"]

for decade, ax in zip(df_winners["Decade"], axs):

    # winner values
    values = (
        df_winners
        .loc[df_winners["Decade"] == decade, winner_region_cols]
        .iloc[0]
        .values
    )

    max_current_values = sum(
        list(df_nominee_melt[df_nominee_melt["Decade"] == decade]["count"].values)
    )

    # total nominations in that decade
    decade_total = (
        df_nominee_melt
        .loc[df_nominee_melt["Decade"] == decade, "count"]
        .sum()
    )

    # height of actual waffle relative to max decade
    y_pos = (max_current_values/max_decade_count) - 0.427

    pie_ax = ax.inset_axes([
        0.10,          # x
        y_pos,  # just above actual decade total
        0.95,          # width
        0.95           # height
    ])

    # print(f"decade={decade}\nmax_current_values = {max_current_values}\nmax_decade_count={max_decade_count}\ny_pos={y_pos}\n")

    pie_ax.pie(
        values,
        colors=colors[:4],
        startangle=90,
        counterclock=False,
        wedgeprops={
            "edgecolor": None,
            "linewidth": 0
        }
    )

    pie_ax.set_aspect("equal")
    pie_ax.axis("off")


# title
fig_text(
  x=0.1,
  y=0.95,
  s=title_main,
  size=60, color=annot_color,
  font=mw_reg,
  highlight_textprops=[
      
  ],
)

# Sub-title
fig_text(
  x=0.1,
  y=0.9,
  s=title_sub,
  size=40, color=annot_color,
  font=mw_i,
  highlight_textprops=[
      
  ],
)

# data source
fig_text(
  x=0.5,
  y=-0.042,
  s=data_source,
  size=15,
  font=mw_reg,
  color=annot_color,
  alpha=0.7,
  ha="center"
)


# One third winners were Asian despite having 14% share in nominations
fig_arrow(
    tail_position=(0.565,0.685), head_position=(0.59, 0.74),
    radius=-0.25, width=2, color=annot_color
)
fig_text(
    x=0.592, y=0.788,
    s="One third winners from <Asia>\nwith only 14% share\nin total nominations.",
    font=mw_i, size=25, ha="left", color=annot_color,
    highlight_textprops=[dict(color=colors[2])]
)
# More than 40% nominations from Europe alone
fig_arrow(
    tail_position=(0.589,0.55), head_position=(0.645, 0.66),
    radius=0.25, width=2, color=annot_color
)
fig_text(
    x=0.62, y=0.692,
    s="More than 40% nominations\nfrom <Europe> alone.",
    font=mw_i, size=25, ha="left", color=annot_color,
    highlight_textprops=[dict(color=colors[3])]
)

# save data story
plt.savefig(os.path.join("..", "viz", save_name), dpi=dpi, pad_inches=1, bbox_inches="tight")